# Trustworthy Smart-Meter Analytics

**Research walkthrough — segmentation, label-free anomaly detection, and uncertainty-aware forecasting**

This notebook is the narrative companion to the tested Python package. It inspects the versioned data and generated experiment artifacts; `src/smart_meter_analytics/` remains the canonical implementation. The central design constraint is simple: no observation from the future test period may influence model fitting or feature statistics.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from smart_meter_analytics.synthetic import validate_data

data = pd.read_csv(ROOT / "data/synthetic_smart_meter_data.csv", parse_dates=["timestamp"])
metrics = json.loads((ROOT / "reports/metrics.json").read_text(encoding="utf-8"))
quality = validate_data(data)
quality

## 1. Data-generating process and integrity

The simulator encodes household-specific scale, four latent usage regimes, morning/evening peaks, weekends, temperature response, solar offsets, and rare injected spikes or drops. Clean counterfactual demand and hidden labels make controlled validation possible. They are not supplied to predictive models.

In [ ]:
summary = pd.DataFrame({
    "value": [
        f"{len(data):,}",
        data["household_id"].nunique(),
        data["timestamp"].nunique(),
        f"{data['true_anomaly'].mean():.3%}",
        data.isna().sum().sum(),
    ]
}, index=["rows", "households", "hourly timestamps", "injected-event rate", "missing values"])
summary

In [ ]:
display(Image(filename=str(ROOT / "figures/load_profiles.png"), width=850))

## 2. Chronological evaluation contract

Every household shares the same time boundaries: the first 70% trains models, the next 10% calibrates the forecast interval, and the final 20% is held out. This prevents the accidental *train-on-some-households, test-on-others* split that row-order slicing can create in panel data. Rolling features are separately grouped by meter and use only prior values.

In [ ]:
pd.Series(metrics["split"], name="experiment protocol")

## 3. Forecasting: benchmarks before complexity

The Random Forest uses causal 1-hour and 24-hour lags, trailing 24-hour statistics, cyclical calendar features, and forecast-available temperature. Persistence and a 24-hour seasonal naive model define the minimum credible comparison. A day-block bootstrap preserves short-range dependence when estimating the MAE interval.

In [ ]:
forecast_table = pd.DataFrame(metrics["forecasting"]["models"]).set_index("model")
forecast_table.round(4)

In [ ]:
display(Image(filename=str(ROOT / "figures/forecast_model_comparison.png"), width=760))
display(Image(filename=str(ROOT / "figures/forecast_actual_vs_predicted.png"), width=1000))

The point forecast reaches 0.0797 kWh MAE, 45.6% below persistence. The separate calibration period produces a nominal 95% split-conformal interval with 94.9% empirical test coverage. The abrupt drop in the trace is an injected event: its miss is diagnostically useful because random events are not forecastable from ordinary lag structure.

## 4. Label-free anomaly detection

Isolation Forest is fitted only to training-period features normalized against each household's typical hourly behavior. Injected labels are used once, after prediction, for held-out evaluation. This avoids converting an unsupervised experiment into a disguised supervised one.

In [ ]:
pd.Series(metrics["anomaly_detection"], name="held-out anomaly result")

In [ ]:
display(Image(filename=str(ROOT / "figures/anomaly_confusion_matrix.png"), width=540))
display(Image(filename=str(ROOT / "figures/anomaly_precision_recall.png"), width=620))

## 5. Segmentation and a useful negative result

K is selected from 2–6 using silhouette score on training-period load-shape features. Hidden simulation regimes are then used for post-hoc audit through adjusted Rand index (ARI) and normalized mutual information—not to select the reported model.

In [ ]:
cluster_sensitivity = pd.read_csv(ROOT / "reports/cluster_selection.csv")
cluster_sensitivity.round(3)

In [ ]:
display(Image(filename=str(ROOT / "figures/customer_clusters.png"), width=1000))

Silhouette narrowly prefers K=2 (0.547), producing a coarse low/high-use partition with ARI 0.312. At the known K=4, silhouette remains 0.541 and ARI becomes 1.000. The inference is not that one answer is universally correct: internal separation and recovery of a domain-defined ontology are different estimands. A real study must define the segmentation purpose before choosing K.

## 6. Interpretation boundary and next experiment

These results validate software and experimental logic under a controlled distribution. They do not establish deployment accuracy, energy theft, customer intent, or causal benefit. The next preregistered experiment should apply this contract to real public meter data using rolling-origin evaluation, missingness and drift stress tests, seasonal coverage, subgroup calibration, and time-series-aware conformal inference.

---
**Full reproduction:** run `python src/generate_smart_meter_data.py --seed 42`, then `python src/train_models.py --seed 42`. Tests in `tests/` enforce deterministic generation, chronological separation, and household-isolated rolling features.